# Robustness check — rolling temporal splits, neural network (AAPL)

Same design as `robustness_nn_spx.ipynb`, applied to AAPL, using the same three splits
as `robustness_aapl.ipynb` (XGBoost). Reuses the fixed architecture already selected for
the main AAPL model (`hidden_layer_sizes=(10,)`, logistic activation, Adam) rather than
re-running the hidden-unit search for each split.

- Same five inputs, same v4 target (`log(C/K)`) as the main model.
- Inputs are standardized on each split's own train+val set (never on data the model
  will be tested on).
- **Before running**: confirm `AAPL_cleaned.csv` covers the full sample (838,276 rows,
  31 Aug 2020 to 29 Aug 2025) — the assert below stops execution otherwise, after the
  earlier robustness check for AAPL initially ran on a truncated file by mistake.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

import joblib


In [ ]:
aapl = pd.read_csv("AAPL_cleaned.csv")
aapl["date"] = pd.to_datetime(aapl["date"])
aapl["exdate"] = pd.to_datetime(aapl["exdate"])

print(aapl.shape)
print(aapl["date"].min(), aapl["date"].max())
aapl.head()


(838276, 19)
2020-08-31 00:00:00 2025-08-29 00:00:00


,secid,date,exdate,strike_price,best_bid,best_offer,volume,open_interest,optionid,close,moneyness (S/K),price,volatility,rate,days_to_maturity,T,last_dividend,frequency,q
0,101594,2020-08-31,2020-09-04,100.00,29.4,29.7,150,411,135212375,129.04,1.290400,29.55,0.375779,0.002148,4,0.010959,0.205,3,0.004766
1,101594,2020-08-31,2020-11-20,80.00,50.1,50.5,0,343,134244951,129.04,1.613000,50.30,0.375779,0.002148,81,0.221918,0.205,3,0.004766
2,101594,2020-08-31,2020-11-20,81.25,48.9,49.3,1,416,134244952,129.04,1.588185,49.10,0.375779,0.002148,81,0.221918,0.205,3,0.004766
3,101594,2020-08-31,2020-11-20,82.50,47.7,48.1,16,383,134244953,129.04,1.564121,47.90,0.375779,0.002148,81,0.221918,0.205,3,0.004766
4,101594,2020-08-31,2020-11-20,83.75,46.5,46.9,0,414,134244954,129.04,1.540776,46.70,0.375779,0.002148,81,0.221918,0.205,3,0.004766


In [ ]:
assert aapl.shape[0] > 800_000, f"Unexpected row count: {aapl.shape[0]:,} — wrong file?"
assert aapl["date"].max().year == 2025, f"Unexpected max date: {aapl['date'].max()} — wrong file?"
print("OK — full AAPL sample loaded.")


OK — full AAPL sample loaded.


## Shared settings (identical to the main NN notebook)

In [ ]:
BINS   = [0, 0.8, 0.95, 1.05, 1.2, 2, 5, 100]
LABELS = ["Deep OTM", "OTM", "ATM", "ITM", "Deep ITM", "Very Deep ITM", "Extreme ITM"]

def bucket_mae(df, preds):
    """MAE by moneyness bucket for BS and each model; last row = all options."""
    t = pd.DataFrame({"bucket": pd.cut(df["moneyness (S/K)"], bins=BINS, labels=LABELS),
                      "BS": np.abs(df["price"].values - df["bs_price"].values)})
    for k, p in preds.items():
        t[k] = np.abs(df["price"].values - np.asarray(p))
    out = t.groupby("bucket", observed=True).mean()
    out.insert(0, "n", t.groupby("bucket", observed=True).size())
    out.loc["ALL"] = [len(t)] + list(t.drop(columns="bucket").mean())
    return out.astype({"n": int})

feature_cols = ["moneyness (S/K)", "T", "rate", "q", "volatility"]
hidden_units = 10  # selected for the main AAPL model; reused as-is, not re-tuned per split


## 1. Black-Scholes benchmark

Identical formula and inputs as the main notebook. BS price does not depend on the
split, so it is computed once on the full dataset.


In [ ]:
d1 = (
    np.log(aapl["close"] / aapl["strike_price"])
    + (aapl["rate"] - aapl["q"] + 0.5 * aapl["volatility"] ** 2) * aapl["T"]
) / (aapl["volatility"] * np.sqrt(aapl["T"]))

d2 = d1 - aapl["volatility"] * np.sqrt(aapl["T"])

aapl["bs_price"] = (
    aapl["close"] * np.exp(-aapl["q"] * aapl["T"]) * norm.cdf(d1)
    - aapl["strike_price"] * np.exp(-aapl["rate"] * aapl["T"]) * norm.cdf(d2)
)

aapl["bs_price"].describe()


,bs_price
count,838276.000000
mean,34.851495
std,42.635250
min,0.000000
25%,1.404682
50%,16.424608
75%,57.070649
max,253.989479


## 2. Three rolling, non-overlapping splits

Same date boundaries used for the XGBoost robustness check on AAPL and for SPX, for
direct comparability.


In [ ]:
splits = {
    "C": {
        "train": ("2020-08-31", "2021-08-31"),
        "val":   ("2021-09-01", "2022-01-01"),
        "test":  ("2022-01-02", "2022-04-30"),
    },
    "B": {
        "train": ("2022-05-01", "2023-05-01"),
        "val":   ("2023-05-02", "2023-09-01"),
        "test":  ("2023-09-02", "2023-12-31"),
    },
    "A": {
        "train": ("2024-01-01", "2025-01-01"),
        "val":   ("2025-01-02", "2025-05-01"),
        "test":  ("2025-05-02", "2025-08-29"),
    },
}

for name, s in splits.items():
    print(name, s)


C {'train': ('2020-08-31', '2021-08-31'), 'val': ('2021-09-01', '2022-01-01'), 'test': ('2022-01-02', '2022-04-30')}
B {'train': ('2022-05-01', '2023-05-01'), 'val': ('2023-05-02', '2023-09-01'), 'test': ('2023-09-02', '2023-12-31')}
A {'train': ('2024-01-01', '2025-01-01'), 'val': ('2025-01-02', '2025-05-01'), 'test': ('2025-05-02', '2025-08-29')}


## 3. Fit and evaluate each split

Same recipe as the main notebook's final NN: standardize inputs on train+val, fit on
`log(C/K)`, evaluate on test, convert back with `exp(pred) * K`. The scaler and the
network are both refit from scratch on each split's own train+val set.


In [ ]:
def fit_eval_split_nn(df, split_dates, label):
    tr = df[(df["date"] >= split_dates["train"][0]) & (df["date"] <= split_dates["train"][1])]
    va = df[(df["date"] >= split_dates["val"][0])   & (df["date"] <= split_dates["val"][1])]
    te = df[(df["date"] >= split_dates["test"][0])  & (df["date"] <= split_dates["test"][1])]
    tv = pd.concat([tr, va]).sort_values("date").reset_index(drop=True)
    te = te.sort_values("date").reset_index(drop=True)

    print(f"[{label}] train {len(tr):,} | val {len(va):,} | train+val {len(tv):,} | test {len(te):,}")
    print(f"[{label}] test window: {te['date'].min()} to {te['date'].max()}")

    X_tv = tv[feature_cols].values
    K_tv = tv["strike_price"].values
    y_tv_log = np.log(tv["price"].values / K_tv)

    scaler = StandardScaler().fit(X_tv)
    X_tv_scaled = scaler.transform(X_tv)

    model = MLPRegressor(
        hidden_layer_sizes=(hidden_units,),
        activation="logistic",
        solver="adam",
        max_iter=2000,
        batch_size=2000,
        early_stopping=True,
        n_iter_no_change=15,
        random_state=42,
    )
    model.fit(X_tv_scaled, y_tv_log)

    X_te = te[feature_cols].values
    K_te = te["strike_price"].values
    X_te_scaled = scaler.transform(X_te)
    test_pred = np.exp(model.predict(X_te_scaled)) * K_te

    return te, test_pred, model, scaler

results = {}
for name, dates in splits.items():
    te, pred, model, scaler = fit_eval_split_nn(aapl, dates, f"AAPL-NN-{name}")
    results[name] = {"test": te, "pred": pred, "model": model, "scaler": scaler}


[AAPL-NN-C] train 191,473 | val 54,919 | train+val 246,392 | test 51,422
[AAPL-NN-C] test window: 2022-01-03 00:00:00 to 2022-04-29 00:00:00
[AAPL-NN-B] train 159,654 | val 48,826 | train+val 208,480 | test 42,995
[AAPL-NN-B] test window: 2023-09-05 00:00:00 to 2023-12-29 00:00:00
[AAPL-NN-A] train 165,967 | val 63,434 | train+val 229,401 | test 59,586
[AAPL-NN-A] test window: 2025-05-02 00:00:00 to 2025-08-29 00:00:00


## 4. Per-bucket MAE, each split

In [ ]:
bucket_tables = {}
for name, r in results.items():
    bucket_tables[name] = bucket_mae(r["test"], {"NN": r["pred"]})
    print(f"--- Split {name} ---")
    print(bucket_tables[name])
    print()


--- Split C ---
                   n        BS         NN
bucket                                   
Deep OTM        9550  0.216962   0.815848
OTM             8267  0.483364   0.720222
ATM             5161  0.855056   1.420693
ITM             6353  1.006506   3.350017
Deep ITM       14103  1.015744   4.841784
Very Deep ITM   7459  0.398344  10.125125
Extreme ITM      529  0.284916  36.267969
ALL            51422  0.667462   3.993486

--- Split B ---
                   n        BS        NN
bucket                                  
Deep OTM        7793  0.120088  0.050674
OTM             7643  0.418076  0.272290
ATM             4863  0.949745  1.464384
ITM             5876  1.046583  2.875413
Deep ITM       11296  0.545793  2.652114
Very Deep ITM   5524  0.177131  4.652276
ALL            42995  0.512694  1.910703

--- Split A ---
                   n        BS         NN
bucket                                   
Deep OTM       13732  2.142575   0.150050
OTM            10398  2.929515   0.

## 5. Side-by-side comparison

In [ ]:
summary = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    summary[f"BS_{name}"] = t["BS"]
    summary[f"NN_{name}"] = t["NN"]

summary.round(3)


,BS_C,NN_C,BS_B,NN_B,BS_A,NN_A
Deep OTM,0.217,0.816,0.120,0.051,2.143,0.150
OTM,0.483,0.720,0.418,0.272,2.930,0.534
ATM,0.855,1.421,0.950,1.464,3.422,1.796
ITM,1.007,3.350,1.047,2.875,2.806,7.041
Deep ITM,1.016,4.842,0.546,2.652,1.235,22.566
Very Deep ITM,0.398,10.125,0.177,4.652,0.282,23.689
Extreme ITM,0.285,36.268,NaN,NaN,0.262,54.977
ALL,0.667,3.993,0.513,1.911,2.028,10.954


In [ ]:
# Same comparison expressed as a ratio (NN error / BS error): <1 means NN wins
ratio = pd.DataFrame(index=LABELS + ["ALL"])
for name, t in bucket_tables.items():
    ratio[name] = t["NN"] / t["BS"]

ratio.round(3)


,C,B,A
Deep OTM,3.760,0.422,0.070
OTM,1.490,0.651,0.182
ATM,1.662,1.542,0.525
ITM,3.328,2.747,2.509
Deep ITM,4.767,4.859,18.272
Very Deep ITM,25.418,26.265,83.936
Extreme ITM,127.294,NaN,209.957
ALL,5.983,3.727,5.402


## 6. Convergence check

Confirms whether the network actually converged on each split's (much smaller) training
set, rather than stopping early for lack of data.


In [ ]:
for name, r in results.items():
    m = r["model"]
    print(f"Split {name}: n_iter_={m.n_iter_} (max_iter={m.max_iter}), "
          f"final loss={m.loss_:.6f}, converged={m.n_iter_ < m.max_iter}")


Split C: n_iter_=164 (max_iter=2000), final loss=0.080386, converged=True
Split B: n_iter_=151 (max_iter=2000), final loss=0.055718, converged=True
Split A: n_iter_=276 (max_iter=2000), final loss=0.152813, converged=True


## 7. Save results

In [ ]:
joblib.dump(
    {"splits": splits, "hidden_units": hidden_units, "summary": summary, "ratio": ratio,
     "bucket_tables": bucket_tables},
    "robustness_nn_aapl_results.pkl"
)


['robustness_nn_aapl_results.pkl']